In [0]:
CREATE SCHEMA IF NOT EXISTS training_catalog.reject;
USE training_catalog.reject;
     

CREATE OR REPLACE TABLE rejected_customers
USING DELTA
AS
SELECT
    *,
    'Missing mandatory customer fields' AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.customers_raw
WHERE CustomerID IS NULL
   OR CustomerName IS NULL
   OR Email IS NULL;
     

CREATE OR REPLACE TABLE rejected_products
USING DELTA
AS
SELECT
    *,
    CASE
        WHEN ProductID IS NULL THEN 'Missing ProductID'
        WHEN ProductName IS NULL THEN 'Missing ProductName'
        WHEN UnitPrice IS NULL THEN 'Missing UnitPrice'
        WHEN CAST(UnitPrice AS DECIMAL(10,2)) <= 0 THEN 'Invalid UnitPrice <= 0'
        ELSE 'Invalid Product'
    END AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.products_raw
WHERE ProductID IS NULL
   OR ProductName IS NULL
   OR UnitPrice IS NULL
   OR CAST(UnitPrice AS DECIMAL(10,2)) <= 0;
     

CREATE OR REPLACE TABLE rejected_stores
USING DELTA
AS
SELECT
    *,
    'Missing store details / region' AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.stores_raw
WHERE StoreID IS NULL
   OR StoreName IS NULL
   OR Region IS NULL;
     

CREATE OR REPLACE TABLE rejected_sales
USING DELTA
AS
SELECT
    *,
    CASE
        WHEN Quantity <= 0 THEN 'Invalid Quantity <= 0'
        WHEN CustomerID IS NULL THEN 'Missing CustomerID'
        WHEN ProductID IS NULL THEN 'Missing ProductID'
        WHEN StoreID IS NULL THEN 'Missing StoreID'
        WHEN TransactionID IS NULL THEN 'Missing TransactionID'
        ELSE 'Invalid Sales Record'
    END AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.sales_raw
WHERE Quantity <= 0
   OR CustomerID IS NULL
   OR ProductID IS NULL
   OR StoreID IS NULL
   OR TransactionID IS NULL;
     

SELECT 'customers' AS table_name, COUNT(*) AS rejected_rows FROM rejected_customers
UNION ALL
SELECT 'products', COUNT(*) FROM rejected_products
UNION ALL
SELECT 'stores', COUNT(*) FROM rejected_stores
UNION ALL
SELECT 'sales', COUNT(*) FROM rejected_sales;
     